# 04 Fashion-MNIST TT-MLP の隣接2 core・1ステップ局所更新

このNotebookは、物理DMRGの基礎から、学習済みFashion-MNIST TT-MLPに対するDMRG-likeな2-core局所更新への接続を確認する。対象は1つの`TTLinear`で、方式Aの局所更新を1回だけ実行し、数式と計算経路が一致するかを見る。

方式Aは、隣接2 coreを結合したテンソル $T$ を独立な学習変数にし、固定minibatchの分類損失から勾配を1回だけ引いて、そのあとSVDで2 coreへ分け直す。DMRGを収束まで回してから別にネットワークを更新する二重ループではない。

今回確認するのは次の対応である。

- 既存のTT-matrix軸順での結合 $G_k, G_{k+1} \to T$
- $T$ を直接使うforwardが、元のcore列のforwardと一致すること
- その経路で $\nabla_T L_B(T)$ が取れ、対象外parameterが動かないこと
- 切り捨てなしのSVDが $T$ を再現し、モデル出力も一致すること
- 同じ $T$ からのrank制限が、Frobeniusノルムの最良近似であること
- 混合正準形のもとで、その局所誤差が重み全体のFrobenius誤差と一致すること
- rank制限後の2 coreを対象の `TTLinear` へ書き戻し、通常の `model(inputs)` がそのcoreで一致すること

複数pairのスイープ、停止判定、複数epochの学習、方式B、通常のTT fine-tuningとの性能比較は、このNotebookの範囲に含めない。それらは将来の `notebooks/40_dmrg/10_fashion_mnist_mlp/` 系列で扱う。損失が下がることや、勾配が非ゼロであることは、一般には成り立たないので検証条件にしない。


## 1. 確認した既存実装と、このNotebookの分担

実装前にリポジトリ上の定義を確認した。推測で名前を置いていない。

| 役割 | 確認した定義 | この実験での使い方 |
| --- | --- | --- |
| TTLinear | `src/nn_compression/compression/tt_linear.py` の `TTLinear` | core shapeは `(r_{k-1}, m_k, n_k, r_k)`。`forward`は`tt_linear_forward` |
| 線形写像 | `tt_matrix.py` の `tt_linear_forward` | denseな $W$ を作らない縮約。結合forwardもこの軸順に合わせる |
| MLP | `models/mlp.py` の `MNISTMLP` | `fc1`: 784→512、`fc2`: 512→256、`fc3`: 256→10。間にReLU |
| checkpoint | `models/30_tt_mps/10_fashion_mnist_mlp/14_rank12_update_scope_ablation/tt_cores_only_selected.pt` | `state_dict`。`fc1.cores.{0,1,2}` と `fc1.bias`、`fc2`、`fc3` |
| mode | Notebook 14が記録した `(32, 8, 2)` と `(2, 8, 49)` | checkpointの `shape[1]`、`shape[2]` と突き合わせてから使う |
| データ | `get_fashion_mnist_datasets`、`split_fashion_mnist_dataset` | seed 0 の 50,000 / 5,000 / 5,000 分割。先頭64件を固定batchにする |
| 変化の確認 | `parameter_change_summary` | 更新していないparameterの完全一致 |
| 3階TTの正準化 | `tt_canonicalize`、`tt_canonicality_errors` | $q_k=m_k n_k$ にまとめてから呼ぶ |

物理DMRGの `merge_two_sites` は3階MPS `(r, n, r)` を4階 `(r, n_{左}, n_{右}, r)` に結合する。`split_two_site_tensor` は分割後のテンソルを単位ノルムへ戻す。TT-matrixは4階で、今回の再分割は重みの値を保つ必要がある。軸順と正規化が違うので、この2関数は使わない。

正準化は流用できる。`tt_matrix_cores_to_tt_cores` は既存の `tt_cores_to_tt_matrix_cores` の逆reshapeで、$q_k$ の並びは `dense_to_tt_matrix_tensor` と同じである。その3階列に `tt_canonicalize` をかけ、4階へ戻す。

共通関数は `src/nn_compression/tt_mps/matrix_two_site.py` に置いた。このNotebookは、対象層・pair・学習率・rank上限・許容誤差と、実行順序、数式、検証結果の表示だけを持つ。


## 2. 実験条件

対象層は既存のFashion-MNIST TT-MLP計画と同じ `fc1` を第一候補にする。隣接pairは設定の `PAIR_INDEX` で選ぶ。既定は先頭のpair（site 0 と site 1）である。

学習率 $\eta$、保持rank $r_{\mathrm{keep}}$、dtype、deviceは関数の内部に書かない。$r_{\mathrm{keep}}$ を `None` にしたときは、正準化後のそのpairのbond次元をNotebook側で読んで関数へ渡す。

固定minibatchは、shuffleするDataLoaderの先頭を `next` で取らない。seed 0 で分けた学習subsetのindex `0 .. BATCH_SIZE-1` を直接読む。


In [34]:
# --- setup: path / imports ---
# OpenMP の二重読み込み警告を避ける（Windows + conda 系でよく出る）。
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import copy
import sys
from pathlib import Path

import torch
import torch.nn.functional as F

# リポジトリ直下の src/ を探して sys.path に入れる。
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    src_dir = candidate / "src"
    if (src_dir / "nn_compression").is_dir():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break
else:
    raise RuntimeError("src/nn_compression が見つかりません。")

# 方式Aの局所更新に使う TT-matrix 2-site API。
from nn_compression.compression import (
    TTLinear,
    assign_tt_linear_cores,
    module_forward_with_merged_tt_linear_pair,
    module_forward_with_tt_linear_cores,
)
from nn_compression.tt_mps import (
    canonicalize_tt_matrix_cores,
    merge_tt_matrix_cores,
    merged_tt_matrix_to_matrix,
    replace_tt_matrix_core_pair,
    split_merged_tt_matrix,
    tt_matrix_canonicality_errors,
    tt_matrix_ranks,
    tt_matrix_to_dense,
)
from nn_compression.datasets import (
    get_fashion_mnist_datasets,
    split_fashion_mnist_dataset,
)
from nn_compression.models import MNISTMLP
from nn_compression.training import parameter_change_summary
from nn_compression.utils import find_project_root, set_named_module, set_seed


In [35]:
# --- 実験条件 ---
SEED = 0
BATCH_SIZE = 64
LAYER_NAME = "fc1"          # 局所更新する TTLinear
PAIR_INDEX = 0              # 隣接 pair の左 site index（0 と 1）
LEARNING_RATE = 1e-3        # T の SGD 1ステップ用 η
# None のときは、正準化後の対象bondを読んで split へ渡す。
R_KEEP = None
# Notebook 14 がこのcheckpointを作ったときのtensorization。
# 読み込んだcoreのshapeと一致しなければ、このまま進めない。
EXPECTED_OUT_MODES = (32, 8, 2)
EXPECTED_IN_MODES = (2, 8, 49)

set_seed(SEED)
project_root = find_project_root(Path.cwd())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint_path = (
    project_root
    / "models"
    / "30_tt_mps"
    / "10_fashion_mnist_mlp"
    / "14_rank12_update_scope_ablation"
    / "tt_cores_only_selected.pt"
)
data_dir = project_root / "data"

# 検証結果を末尾でまとめて表示するための記録用リスト。
checks = []


def record(name, status, detail):
    """検証項目を記録し、失敗ならその場で止める。"""
    checks.append({"item": name, "status": status, "detail": detail})
    print(f"[{status}] {name}")
    print(f"  {detail}")
    if status == "検証失敗":
        raise AssertionError(f"{name}: {detail}")


def comparison_tolerances(dtype):
    """縮約・logits・直交誤差に使う許容誤差。

    float32の機械精度は約1.2e-7。3 coreの縮約と、その後のReLU・fc2・fc3で
    誤差が積もる。既存DMRGテスト `tests/_dmrg_testing.py` のfloat32局所演算は
    atol=rtol=2e-4 を使っている。ここでも同じ桁を使う。
    float64は、同じテストの桁に合わせて atol=rtol=1e-8 にする。
    """
    if dtype == torch.float32:
        return {"rtol": 2e-4, "atol": 2e-4}
    if dtype == torch.float64:
        return {"rtol": 1e-8, "atol": 1e-8}
    raise TypeError(f"未対応のdtypeです: {dtype}")


def svd_absolute_tolerance(matrix):
    """還元SVDの再構成誤差に使う絶対許容。

    floatの機械精度を eps、行列のFrobeniusノルムを scale、大きい方の次元を
    n とすると、実用上の上界を 50 * eps * max(scale, 1) * n とする。
    係数50は、次元が数十から数百の行列で積和が積もる分の余裕である。
    """
    eps = torch.finfo(matrix.dtype).eps
    scale = float(torch.linalg.vector_norm(matrix.detach()))
    rows, cols = matrix.shape
    return 50.0 * eps * max(scale, 1.0) * max(rows, cols)


print("device:", DEVICE)
print("checkpoint:", checkpoint_path)
print("exists:", checkpoint_path.is_file())


device: cuda
checkpoint: D:\dev\nn-compression-svd-dmrg\models\30_tt_mps\10_fashion_mnist_mlp\14_rank12_update_scope_ablation\tt_cores_only_selected.pt
exists: True


## 3. 共通の初期状態

checkpointの `state_dict` から `MNISTMLP` を復元する。`out_modes` と `in_modes` はstate_dictに入っていないので、coreの `shape[1]` と `shape[2]` から読み、Notebook 14の記録 `(32, 8, 2)`、`(2, 8, 49)` と一致することを確認する。

$G_k$ の軸は実装どおり

$$
G_k \in \mathbb{R}^{r_{k-1} \times m_k \times n_k \times r_k}
$$

である。`fc1` の3 coreは、この順で `fc1.cores.0`、`fc1.cores.1`、`fc1.cores.2` に入っている。

固定minibatch $B$ に対する損失は、`F.cross_entropy` の既定（batch平均）と同じである。

$$
L_B(\theta)
=
\frac{1}{|B|}
\sum_{s \in B}
\mathrm{CE}(f(x_s; \theta), y_s)
$$

このセルでは、checkpointを読んだ直後のlogitsと $L_B$ を記録し、parameterの複製を残す。このforwardは記録用なので `torch.no_grad` で行う。学習用のグラフは、あとで $T$ を通すforwardで別に作る。


In [36]:
# --- 共通の初期状態: checkpoint → モデル / 固定minibatch ---
if not checkpoint_path.is_file():
    record(
        "checkpointの存在",
        "未実行",
        f"{checkpoint_path} がありません。代替モデルでは実行しません。",
    )
    raise FileNotFoundError(checkpoint_path)

# state_dict から TT core の mode を読む（checkpoint 自体に mode は入っていない）。
state = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
core_keys = sorted(
    (key for key in state if key.startswith(f"{LAYER_NAME}.cores.")),
    key=lambda key: int(key.rsplit(".", 1)[-1]),
)
if not core_keys or f"{LAYER_NAME}.bias" not in state:
    record(
        "checkpointの形式",
        "検証失敗",
        f"{LAYER_NAME} の cores と bias が state_dict にありません: {sorted(state)}",
    )

import math

out_modes = tuple(int(state[key].shape[1]) for key in core_keys)
in_modes = tuple(int(state[key].shape[2]) for key in core_keys)
dtype = state[core_keys[0]].dtype
if out_modes != EXPECTED_OUT_MODES or in_modes != EXPECTED_IN_MODES:
    record(
        "modeとcheckpointの対応",
        "検証失敗",
        f"core shapeから読んだ mode は out={out_modes}, in={in_modes}。"
        f"Notebook 14の記録は out={EXPECTED_OUT_MODES}, in={EXPECTED_IN_MODES}。",
    )
if math.prod(out_modes) != 512 or math.prod(in_modes) != 784:
    record(
        "fc1の入出力次元",
        "検証失敗",
        f"prod(out_modes)={math.prod(out_modes)}, prod(in_modes)={math.prod(in_modes)}",
    )

# MNISTMLP の fc1 を TTLinear に差し替えてから state_dict を載せる。
dummy_cores = [
    torch.zeros(state[key].shape, dtype=dtype, device=DEVICE) for key in core_keys
]
model = MNISTMLP().to(device=DEVICE, dtype=dtype)
set_named_module(
    model,
    LAYER_NAME,
    TTLinear(dummy_cores, out_modes, in_modes, bias=torch.zeros_like(state[f"{LAYER_NAME}.bias"])),
)
model.load_state_dict(state)
model.eval()
layer = model.get_submodule(LAYER_NAME)

# 固定minibatch: shuffle せず index 0..BATCH_SIZE-1 を直接読む。
full_train, _test_dataset = get_fashion_mnist_datasets(data_dir, download=False)
train_dataset, _validation_dataset, _rank_dataset = split_fashion_mnist_dataset(
    full_train,
    (50_000, 5_000, 5_000),
    seed=SEED,
)
images = torch.stack([train_dataset[index][0] for index in range(BATCH_SIZE)])
labels = torch.tensor([train_dataset[index][1] for index in range(BATCH_SIZE)])
images = images.to(device=DEVICE, dtype=dtype)
labels = labels.to(device=DEVICE)

# 記録用の基準 logits / loss（学習グラフは切る）。
with torch.no_grad():
    logits_checkpoint = model(images)
    loss_checkpoint = F.cross_entropy(logits_checkpoint, labels)
checkpoint_state = copy.deepcopy(model.state_dict())
cores_checkpoint = [core.detach().clone() for core in layer.cores]

print("dtype:", dtype)
print("layer:", LAYER_NAME, type(layer).__name__)
print("out_modes:", layer.out_modes)
print("in_modes:", layer.in_modes)
print("ranks:", tt_matrix_ranks(list(layer.cores)))
print("core shapes:", [tuple(core.shape) for core in layer.cores])
print("batch:", tuple(images.shape), "labels:", tuple(labels.shape))
print("checkpoint loss:", float(loss_checkpoint))
record(
    "初期状態の準備",
    "検証成功",
    f"dtype={dtype}, device={DEVICE}, ranks={tt_matrix_ranks(list(layer.cores))}, "
    f"loss={float(loss_checkpoint):.6f}, finite={bool(torch.isfinite(logits_checkpoint).all())}",
)


dtype: torch.float32
layer: fc1 TTLinear
out_modes: (32, 8, 2)
in_modes: (2, 8, 49)
ranks: (1, 12, 12, 1)
core shapes: [(1, 32, 2, 12), (12, 8, 8, 12), (12, 2, 49, 1)]
batch: (64, 1, 28, 28) labels: (64,)
checkpoint loss: 0.14065685868263245
[検証成功] 初期状態の準備
  dtype=torch.float32, device=cuda, ranks=(1, 12, 12, 1), loss=0.140657, finite=True


## 4. 混合正準形

pairの左siteを直交中心にする。`center = PAIR_INDEX` なので、

- indexが `center` より小さいcoreは左直交である。左展開 $(r_{k-1} m_k n_k,\ r_k)$ の列が正規直交で、$\lVert M^{\mathsf T} M - I \rVert_F$ が0に近い。
- indexが `center` より大きいcoreは右直交である。右展開 $(r_{k-1},\ m_k n_k r_k)$ の行が正規直交で、$\lVert N N^{\mathsf T} - I \rVert_F$ が0に近い。

`PAIR_INDEX = 0` では、pairの左にcoreがない。pairの右core（site 1）と、そのさらに右（site 2）が右直交になる。pairの左coreは直交中心であり、それ自身は左直交とは限らない。

このgaugeはQRによる変換なので、dense重み

$$
W = \mathrm{reshape}(\mathrm{contract}(G_1,\ldots,G_d))
$$

と、モデル出力は変わらない。確認に使う `tt_matrix_to_dense` は診断用であり、このあとの学習forwardでは呼ばない。

正準化のあと、pairの外側は等長写像である。したがって結合テンソル $T$ のFrobeniusノルムは $W$ のFrobeniusノルムと一致し、あとで $T$ を低rank近似したときの局所誤差は $W$ の誤差と一致する。正準化していない列では、この一致は主張しない。$T \mapsto W$ は入力に依らない重みの写像で、$T \mapsto \mathrm{logits}$ はminibatchに依存する写像である。この2つは別に確認する。


In [37]:
# --- 混合正準形: pair 左 site を直交中心にする ---
center = PAIR_INDEX
if not 0 <= center < len(layer.cores) - 1:
    record(
        "pairの範囲",
        "検証失敗",
        f"PAIR_INDEX={center} は core数 {len(layer.cores)} の隣接pairではありません。",
    )

# gauge 変換なので dense W と logits は不変であるはず。
weight_before = tt_matrix_to_dense(cores_checkpoint, layer.out_modes, layer.in_modes)
canonical_cores = canonicalize_tt_matrix_cores(
    cores_checkpoint,
    layer.out_modes,
    layer.in_modes,
    center,
)
assign_tt_linear_cores(layer, canonical_cores)
weight_after = tt_matrix_to_dense(list(layer.cores), layer.out_modes, layer.in_modes)
tolerances = comparison_tolerances(dtype)

try:
    torch.testing.assert_close(weight_after, weight_before, **tolerances)
except AssertionError as error:
    record("正準化で重みが不変", "検証失敗", str(error))
weight_gap = float((weight_after - weight_before).detach().abs().max())
record(
    "正準化で重みが不変",
    "検証成功",
    f"max_abs={weight_gap:.3e}, tolerances={tolerances}",
)

with torch.no_grad():
    logits_canonical = model(images)
    loss_canonical = F.cross_entropy(logits_canonical, labels)
try:
    torch.testing.assert_close(logits_canonical, logits_checkpoint, **tolerances)
except AssertionError as error:
    record("正準化でlogitsが不変", "検証失敗", str(error))
record(
    "正準化でlogitsが不変",
    "検証成功",
    f"logits max_abs={float((logits_canonical - logits_checkpoint).abs().max()):.3e}, "
    f"loss {float(loss_checkpoint):.6f} -> {float(loss_canonical):.6f}",
)

# center より左は左直交、右は右直交になっているか。
errors = tt_matrix_canonicality_errors(
    list(layer.cores),
    layer.out_modes,
    layer.in_modes,
    center,
)
ortho_limit = tolerances["atol"]
left_values = [float(error.detach()) for error in errors["left"]]
right_values = [float(error.detach()) for error in errors["right"]]
left_ok = all(value <= ortho_limit for value in left_values)
right_ok = all(value <= ortho_limit for value in right_values) and len(right_values) == len(layer.cores) - center - 1
record(
    "左直交",
    "検証成功" if left_ok else "検証失敗",
    f"center={center} より左の ||M^T M - I||_F = {left_values}（空なら、pairの左にcoreがない）。上限 {ortho_limit:.1e}",
)
record(
    "右直交",
    "検証成功" if right_ok else "検証失敗",
    f"centerより右の ||N N^T - I||_F = {right_values}。上限 {ortho_limit:.1e}",
)

# 以後の「対象外parameterが不変」比較の基準。
state_after_canon = copy.deepcopy(model.state_dict())
bond_original = int(layer.cores[PAIR_INDEX].shape[-1])
print("bond after canonicalization:", bond_original)
print("core shapes:", [tuple(core.shape) for core in layer.cores])


[検証成功] 正準化で重みが不変
  max_abs=6.258e-07, tolerances={'rtol': 0.0002, 'atol': 0.0002}
[検証成功] 正準化でlogitsが不変
  logits max_abs=2.289e-05, loss 0.140657 -> 0.140657
[検証成功] 左直交
  center=0 より左の ||M^T M - I||_F = []（空なら、pairの左にcoreがない）。上限 2.0e-04
[検証成功] 右直交
  centerより右の ||N N^T - I||_F = [6.406932016034261e-07, 7.152992793635349e-07]。上限 2.0e-04
bond after canonicalization: 12
core shapes: [(1, 32, 2, 12), (12, 8, 8, 12), (12, 2, 49, 1)]


## 5. 2 coreを結合して、独立な学習変数 $T$ にする

pair $(k, k+1)$ のcoreを、共有bond $\beta$ で縮約する。軸はcoreの保存順のままである。

$$
\begin{aligned}
G_k[\alpha, i_k, j_k, \beta]
&\in \mathbb{R}^{r_{k-1} \times m_k \times n_k \times r_k}, \\
G_{k+1}[\beta, i_{k+1}, j_{k+1}, \gamma]
&\in \mathbb{R}^{r_k \times m_{k+1} \times n_{k+1} \times r_{k+1}}, \\
T[\alpha, i_k, j_k, i_{k+1}, j_{k+1}, \gamma]
&=
\sum_{\beta=1}^{r_k}
G_k[\alpha, i_k, j_k, \beta]\,
G_{k+1}[\beta, i_{k+1}, j_{k+1}, \gamma].
\end{aligned}
$$

$T$ のshapeは $(r_{k-1}, m_k, n_k, m_{k+1}, n_{k+1}, r_{k+1})$ である。

`merge_tt_matrix_cores` はグラフを切らない。学習変数は、この戻り値をNotebook側で `detach().clone()` し、`requires_grad=True` にしたleafである。元の $G_k$ と $G_{k+1}$ の計算履歴にはつながっていない。

対象外のcore、bias、他の層は `requires_grad=False` にする。$T$ はモジュールのparameterではないので、この凍結は $T$ にはかからない。


In [38]:
# --- 隣接2 core を結合し、独立な学習変数 T にする ---
# 方式Aでは T だけを動かすので、モジュール側の parameter はすべて凍結。
for parameter in model.parameters():
    parameter.requires_grad_(False)

left_core = layer.cores[PAIR_INDEX]
right_core = layer.cores[PAIR_INDEX + 1]
# 共有 bond β で縮約 → 6階テンソル。
merged_from_cores = merge_tt_matrix_cores(left_core, right_core)
# 元の core グラフから切り離した leaf を学習変数にする。
T = merged_from_cores.detach().clone().requires_grad_(True)

print("T.shape:", tuple(T.shape))
print("T is leaf:", T.is_leaf, "requires_grad:", T.requires_grad)
print("frozen parameters:", sum(not p.requires_grad for p in model.parameters()), "/", sum(1 for _ in model.parameters()))
record(
    "独立な学習変数T",
    "検証成功" if T.is_leaf and T.requires_grad and T.shape == merged_from_cores.shape else "検証失敗",
    f"shape={tuple(T.shape)}, leaf={T.is_leaf}",
)


T.shape: (1, 32, 2, 8, 8, 12)
T is leaf: True requires_grad: True
frozen parameters: 8 / 8
[検証成功] 独立な学習変数T
  shape=(1, 32, 2, 8, 8, 12), leaf=True


## 6. $T$ を直接使うforward

`tt_linear_forward` は、site $k$ に入る直前の状態を

$$
(\mathrm{batch}, i_1, \ldots, i_{k-1}, \alpha_{k-1}, j_k, \ldots, j_d)
$$

という軸で持つ。core $( \alpha_{k-1}, i_k, j_k, \alpha_k )$ と、$\alpha_{k-1}$ と $j_k$ を縮約する。

pairでは、2 coreの代わりに $T$ の軸

$$
(\alpha,\ i_k,\ j_k,\ i_{k+1},\ j_{k+1},\ \gamma)
$$

を一度に縮約する。縮約するのは $\alpha$, $j_k$, $j_{k+1}$ で、残る $(i_k, i_{k+1}, \gamma)$ を $\alpha$ があった位置へ戻す。その直後の軸は、通常のforwardがsite $k+1$ を終えた状態と揃う。pair以外のcoreとbias、ReLU、`fc2`、`fc3` はモデルが持つ実装のままである。

この学習用forwardは `tt_matrix_to_dense` を呼ばない。$T$ がちょうど $G_k$ と $G_{k+1}$ の縮約なら、logitsは元のcore列のlogitsと、上で定めた許容誤差の範囲で一致する。


In [39]:
# --- T を直接使う forward（左ブロック / T / 右ブロック） ---
# pair 以外の core・bias・ReLU・後続層はそのまま。dense W は作らない。
logits_T = module_forward_with_merged_tt_linear_pair(
    model,
    images,
    LAYER_NAME,
    PAIR_INDEX,
    T,
)
loss_T = F.cross_entropy(logits_T, labels)
try:
    torch.testing.assert_close(logits_T, logits_canonical, **tolerances)
except AssertionError as error:
    record("Tを使うforwardとcore列forwardの一致", "検証失敗", str(error))
record(
    "Tを使うforwardとcore列forwardの一致",
    "検証成功",
    f"max_abs={float((logits_T - logits_canonical).detach().abs().max()):.3e}, "
    f"grad_fn={type(logits_T.grad_fn).__name__}, loss={float(loss_T.detach()):.6f}",
)
# 次の backward のために、T → logits のグラフが残っていることを確認。
if logits_T.grad_fn is None:
    record("Tからlogitsまでの計算グラフ", "検証失敗", "logits_T.grad_fn がありません。")
record(
    "Tからlogitsまでの計算グラフ",
    "検証成功",
    f"grad_fn={type(logits_T.grad_fn).__name__}",
)


[検証成功] Tを使うforwardとcore列forwardの一致
  max_abs=9.537e-06, grad_fn=AddmmBackward0, loss=0.140657
[検証成功] Tからlogitsまでの計算グラフ
  grad_fn=AddmmBackward0


## 7. 勾配1ステップ

固定minibatchに対する学習用の損失は、直前セルの $L_B(T)$ である。

$$
g = \nabla_T L_B(T),
\qquad
T_{\mathrm{new}} = T - \eta g.
$$

optimizerは使わない。momentumもweight decayもない。更新するのは $T$ だけである。

$T_{\mathrm{new}}$ を作ったあと `detach` するのは、記録用forwardとSVDを学習グラフの続きにしないためである。backwardは、その `detach` より前に終わっている。

一般に損失が下がるとは限らないので、更新後の損失は記録するだけで、減少は要求しない。$g=0$ も失敗にはしない。対象外parameterは、正準化直後に保存したstateと完全一致することを見る。


In [40]:
# --- 勾配1ステップ: T_new = T - η ∇_T L_B(T) ---
loss_T.backward()
g = T.grad
grad_ok = (
    g is not None
    and tuple(g.shape) == tuple(T.shape)
    and bool(torch.isfinite(g).all())
)
record(
    "勾配gの取得",
    "検証成功" if grad_ok else "検証失敗",
    "gがありません。" if g is None else f"shape={tuple(g.shape)}, finite={bool(torch.isfinite(g).all())}",
)

# optimizer は使わず、素の SGD を1回だけ。
grad_norm = float(torch.linalg.vector_norm(g))
T_step = T - LEARNING_RATE * g
# 以降の記録用 forward / SVD を学習グラフの続きにしない。
T_new = T_step.detach()
step_norm = float(torch.linalg.vector_norm(T_new - T.detach()))
values_ok = bool(torch.isfinite(T_new).all()) and bool(torch.isfinite(torch.tensor(grad_norm)))
record(
    "SGD 1ステップの有限性",
    "検証成功" if values_ok else "検証失敗",
    f"||g||_F={grad_norm:.6e}, ||T_new-T||_F={step_norm:.6e}, eta={LEARNING_RATE}",
)

# 凍結したはずの parameter に grad が付いていないこと、値が変わっていないこと。
model_grads = [name for name, parameter in model.named_parameters() if parameter.grad is not None]
rows = parameter_change_summary(state_after_canon, model)
unchanged = all(row["exactly_equal"] for row in rows) and not model_grads
record(
    "対象外parameterが不変",
    "検証成功" if unchanged else "検証失敗",
    f"parameter数={len(rows)}, 値の変化あり={sum(not row['exactly_equal'] for row in rows)}, "
    f"gradが付いたparameter={model_grads}",
)

# 損失の減少は要求しない（一般には保証されない）。記録のみ。
with torch.no_grad():
    logits_new = module_forward_with_merged_tt_linear_pair(
        model,
        images,
        LAYER_NAME,
        PAIR_INDEX,
        T_new,
    )
    loss_new = F.cross_entropy(logits_new, labels)
print("loss before step:", float(loss_T.detach()))
print("loss after step (record forward):", float(loss_new))
print("delta loss:", float(loss_new - loss_T.detach()))
record(
    "更新後損失の再計算",
    "検証成功" if bool(torch.isfinite(loss_new)) else "検証失敗",
    f"L(T)={float(loss_T.detach()):.6f}, L(T_new)={float(loss_new):.6f}. "
    "減少は要求していない。このforwardはno_gradの記録用である。",
)


[検証成功] 勾配gの取得
  shape=(1, 32, 2, 8, 8, 12), finite=True
[検証成功] SGD 1ステップの有限性
  ||g||_F=3.077406e-01, ||T_new-T||_F=3.077670e-04, eta=0.001
[検証成功] 対象外parameterが不変
  parameter数=8, 値の変化あり=0, gradが付いたparameter=[]
loss before step: 0.14065702259540558
loss after step (record forward): 0.1405630111694336
delta loss: -9.401142597198486e-05
[検証成功] 更新後損失の再計算
  L(T)=0.140657, L(T_new)=0.140563. 減少は要求していない。このforwardはno_gradの記録用である。


## 8. 切り捨てなしのSVD再分割

更新後の $T_{\mathrm{new}}$ を、左3軸と右3軸で行列にする。

$$
\begin{aligned}
M[(\alpha, i_k, j_k),\ (i_{k+1}, j_{k+1}, \gamma)]
&=
T_{\mathrm{new}}[\alpha, i_k, j_k, i_{k+1}, j_{k+1}, \gamma], \\
M &= U \Sigma V^{\mathsf T}.
\end{aligned}
$$

$M$ のshapeは $(r_{k-1} m_k n_k,\ m_{k+1} n_{k+1} r_{k+1})$ である。使うのは `full_matrices=False` の還元SVDで、返った成分は全部保持する。非ゼロ特異値の本数を `matrix_rank` で推定して削る処理は、この確認には混ぜない。

モデルへ戻すcoreは、checkpointと同じfloat32のSVDで作る。float32の再構成残差は、この行列では切り捨て誤差より大きくなり得る。等式 $M=U\Sigma V^{\mathsf T}$ の確認は、同じ $T_{\mathrm{new}}$ をfloat64へ移したSVDで行う。float32側では、tailが空であること、左coreの直交、logitsの一致を見る。

左から右への分割は

$$
G_k' = \mathrm{reshape}(U),
\qquad
G_{k+1}' = \mathrm{reshape}(\Sigma V^{\mathsf T})
$$

である。特異値をノルムで割らない。したがって再結合は $T_{\mathrm{new}}$ を再現し、そのcore列のモデル出力と損失は、$T_{\mathrm{new}}$ を直接入れた記録用forwardと一致する。

この分割では $U$ の列が正規直交なので、$G_k'$ の左展開は左直交になる。$\Sigma$ は右coreに入るので、$G_{k+1}'$ が右直交であることは要求しない。


In [41]:
# --- 切り捨てなし SVD: T_new → (G_k', G_{k+1}') ---
# 左から右: G_k' = reshape(U), G_{k+1}' = reshape(Σ V^T)。ノルム再正規化はしない。
split_full = split_merged_tt_matrix(T_new, max_bond=None)
restored = merge_tt_matrix_cores(split_full.left_core, split_full.right_core)
matrix_new = merged_tt_matrix_to_matrix(T_new)
matrix_restored = merged_tt_matrix_to_matrix(restored)
residual32 = float(torch.linalg.vector_norm(matrix_new - matrix_restored))

# 恒等式の確認は float64。float32 の再構成残差は、このサイズでは δ_r を隠す。
T_new_64 = T_new.detach().to(dtype=torch.float64)
split_full_64 = split_merged_tt_matrix(T_new_64, max_bond=None)
restored_64 = merge_tt_matrix_cores(split_full_64.left_core, split_full_64.right_core)
matrix_new_64 = merged_tt_matrix_to_matrix(T_new_64)
residual64 = float(
    torch.linalg.vector_norm(matrix_new_64 - merged_tt_matrix_to_matrix(restored_64))
)
scale64 = float(torch.linalg.vector_norm(matrix_new_64))
# float64 の機械精度は約 2e-16。64×768 の還元SVDでも、相対残差は 1e-12 を
# 十分下回る。上限は scale * 1e-10 とし、空の tail とセットで等式を確認する。
residual64_limit = max(scale64, 1.0) * 1e-10
print("full retained rank:", split_full.retained_rank)
print("float32 reconstruction residual:", residual32)
print("float64 reconstruction residual:", residual64, "limit:", residual64_limit)
print("float64 discarded squared norm:", float(split_full_64.discarded_squared_norm))
record(
    "切り捨てなしSVDの再結合",
    "検証成功"
    if residual64 <= residual64_limit and float(split_full_64.discarded_squared_norm) == 0.0
    else "検証失敗",
    f"float64 ||M-M_hat||_F={residual64:.3e}, 上限={residual64_limit:.3e}, "
    f"捨てた特異値の二乗和={float(split_full_64.discarded_squared_norm):.3e}。"
    f"float32の再構成残差は {residual32:.3e} で、これは丸めの観測であり等式の判定には使わない。",
)

# U の列が正規直交 → 左 core は左直交。右 core は Σ を含むので右直交は要求しない。
left_matrix = split_full.left_core.reshape(-1, split_full.left_core.shape[-1])
identity = torch.eye(left_matrix.shape[1], dtype=left_matrix.dtype, device=left_matrix.device)
left_ortho = float(torch.linalg.vector_norm(left_matrix.mT @ left_matrix - identity))
record(
    "再分割した左coreの左直交",
    "検証成功" if left_ortho <= tolerances["atol"] else "検証失敗",
    f"||U^T U - I||_F={left_ortho:.3e}。右coreはΣを含むので右直交は要求しない。",
)

# 分け直した core で forward し、T_new 直接経路と一致するか。
cores_full = replace_tt_matrix_core_pair(
    list(layer.cores),
    PAIR_INDEX,
    split_full.left_core,
    split_full.right_core,
)
with torch.no_grad():
    logits_full = module_forward_with_tt_linear_cores(model, images, LAYER_NAME, cores_full)
    loss_full = F.cross_entropy(logits_full, labels)
try:
    torch.testing.assert_close(logits_full, logits_new, **tolerances)
except AssertionError as error:
    record("再分割後のlogits", "検証失敗", str(error))
loss_gap = abs(float(loss_full - loss_new))
record(
    "再分割後のlogitsと損失",
    "検証成功" if loss_gap <= tolerances["atol"] else "検証失敗",
    f"logits max_abs={float((logits_full - logits_new).abs().max()):.3e}, "
    f"|loss差|={loss_gap:.3e}。"
    f"logitsの許容は atol={tolerances['atol']:.1e} に加え rtol={tolerances['rtol']:.1e} * |logit|。"
    f"損失の許容は絶対値 {tolerances['atol']:.1e}。",
)


full retained rank: 64
float32 reconstruction residual: 0.0012176439631730318
float64 reconstruction residual: 2.71881048123284e-12 limit: 1.1972044224171704e-08
float64 discarded squared norm: 0.0
[検証成功] 切り捨てなしSVDの再結合
  float64 ||M-M_hat||_F=2.719e-12, 上限=1.197e-08, 捨てた特異値の二乗和=0.000e+00。float32の再構成残差は 1.218e-03 で、これは丸めの観測であり等式の判定には使わない。
[検証成功] 再分割した左coreの左直交
  ||U^T U - I||_F=7.491e-05。右coreはΣを含むので右直交は要求しない。
[検証成功] 再分割後のlogitsと損失
  logits max_abs=4.082e-04, |loss差|=2.384e-07。logitsの許容は atol=2.0e-04 に加え rtol=2.0e-04 * |logit|。損失の許容は絶対値 2.0e-04。


## 9. rankを制限したSVD再分割

前節のcoreをさらに削るのではなく、同じ $T_{\mathrm{new}}$ から独立に、rank $r_{\mathrm{keep}}$ だけを残す。

$$
M_r = U_r \Sigma_r V_r^{\mathsf T},
\qquad
\delta_r^2
=
\lVert M - M_r \rVert_F^2
=
\sum_{j > r_{\mathrm{keep}}} \sigma_j^2.
$$

$\delta_r^2$ は、再結合との残差から直接計算した値と、捨てた特異値から計算した値の両方を記録する。この恒等式の確認はfloat64のSVDで行う。float32のまま残差を作ると、還元SVDの再構成誤差が $\delta_r$ より大きくなり、2つの値が別の量を見てしまう。モデルのcoreと重み誤差は、これまでどおりfloat32の分割を使う。

この切り捨ては、行列 $M$ のFrobeniusノルムに関する最良低rank近似である。分類損失 $L_B$ を最小化する操作ではない。したがって、切り捨て前後の損失変化は診断値として記録し、改善したことや悪化したことを結論にはしない。

混合正準形が前節まで維持されているので、局所誤差 $\delta_r = \lVert T_{\mathrm{new}} - T_r \rVert_F$ は、同じ2つのcore列から診断用に再構成した重みの $\lVert W(T_{\mathrm{new}}) - W(T_r) \rVert_F$ と一致する。これは $T \mapsto W$ の確認である。logitsの差は、入力に依存する $T \mapsto f(x)$ の確認として別に見る。

この節の `cores_truncated` は一時forwardに渡すだけで、ここでは `model` の Parameter へは書き戻さない。書き戻しは次節で行う。


In [42]:
# --- rank 制限 SVD: 同じ T_new から独立に r_keep だけ残す ---
# 前節の core をさらに削るのではなく、同じ T_new からやり直す。
r_keep = bond_original if R_KEEP is None else int(R_KEEP)
split_truncated = split_merged_tt_matrix(T_new, max_bond=r_keep)
split_truncated_64 = split_merged_tt_matrix(T_new_64, max_bond=r_keep)
truncated_64 = merge_tt_matrix_cores(
    split_truncated_64.left_core,
    split_truncated_64.right_core,
)
# ||M - M_r||_F^2 と Σ_{j>r} σ_j^2 が一致するか（float64 で確認）。
residual_sq_64 = (
    matrix_new_64 - merged_tt_matrix_to_matrix(truncated_64)
).square().sum()
tail_sq_64 = split_truncated_64.discarded_squared_norm
try:
    torch.testing.assert_close(residual_sq_64, tail_sq_64, rtol=1e-8, atol=1e-12)
except AssertionError as error:
    record("切り捨て誤差の2通りの一致", "検証失敗", str(error))
delta_r = float(split_truncated.discarded_squared_norm.sqrt())
delta_r_64 = float(tail_sq_64.sqrt())
record(
    "切り捨て誤差の2通りの一致",
    "検証成功",
    f"float64 残差 ||M-M_r||_F^2={float(residual_sq_64):.6e}, "
    f"特異値 Σ_{{j>r}} σ_j^2={float(tail_sq_64):.6e}, δ_r(float64)={delta_r_64:.6e}。"
    f"モデルに使う float32 分割の δ_r={delta_r:.6e}。"
    "許容は rtol=1e-8, atol=1e-12。float64の機械精度に対するこの行列サイズの余裕である。",
)

cores_truncated = replace_tt_matrix_core_pair(
    list(layer.cores),
    PAIR_INDEX,
    split_truncated.left_core,
    split_truncated.right_core,
)
with torch.no_grad():
    # 正準化後なら局所 δ_r と重み全体の Frobenius 誤差が一致する。
    weight_exact = tt_matrix_to_dense(cores_full, layer.out_modes, layer.in_modes)
    weight_truncated = tt_matrix_to_dense(cores_truncated, layer.out_modes, layer.in_modes)
    logits_truncated = module_forward_with_tt_linear_cores(
        model,
        images,
        LAYER_NAME,
        cores_truncated,
    )
    loss_truncated = F.cross_entropy(logits_truncated, labels)
global_weight_error = float(torch.linalg.vector_norm(weight_exact - weight_truncated))
try:
    torch.testing.assert_close(
        torch.tensor(global_weight_error, dtype=torch.float64),
        torch.tensor(delta_r, dtype=torch.float64),
        rtol=1e-3,
        atol=1e-4,
    )
except AssertionError as error:
    record("局所Frobenius誤差と重み誤差", "検証失敗", str(error))
record(
    "局所Frobenius誤差と重み誤差",
    "検証成功",
    f"||W(T_new)-W(T_r)||_F={global_weight_error:.6e}, δ_r={delta_r:.6e}. "
    "これは正準化後の T→W の対応であり、logitsの対応ではない。",
)

print("bond original:", bond_original)
print("bond without truncation:", split_full.retained_rank)
print("bond after truncation:", split_truncated.retained_rank)
print("absolute truncation error δ_r:", delta_r)
relative = delta_r / max(float(torch.linalg.vector_norm(matrix_new)), 1e-30)
print("relative truncation error δ_r / ||M||_F:", relative)
print("loss T_new:", float(loss_new))
print("loss truncated:", float(loss_truncated))
print("loss change:", float(loss_truncated - loss_new))
record(
    "rank制限の記録",
    "検証成功",
    f"元bond={bond_original}, 切り捨てなし={split_full.retained_rank}, "
    f"切り捨て後={split_truncated.retained_rank}, "
    f"絶対誤差={delta_r:.6e}, 相対誤差={relative:.6e}, "
    f"損失変化={float(loss_truncated - loss_new):.6e}. "
    "切り捨てはFrobenius最良近似であり、分類損失の最小化ではない。",
)


[検証成功] 切り捨て誤差の2通りの一致
  float64 残差 ||M-M_r||_F^2=7.910722e-08, 特異値 Σ_{j>r} σ_j^2=7.910722e-08, δ_r(float64)=2.812601e-04。モデルに使う float32 分割の δ_r=2.813202e-04。許容は rtol=1e-8, atol=1e-12。float64の機械精度に対するこの行列サイズの余裕である。
[検証成功] 局所Frobenius誤差と重み誤差
  ||W(T_new)-W(T_r)||_F=2.819219e-04, δ_r=2.813202e-04. これは正準化後の T→W の対応であり、logitsの対応ではない。
bond original: 12
bond without truncation: 64
bond after truncation: 12
absolute truncation error δ_r: 0.00028132018633186817
relative truncation error δ_r / ||M||_F: 2.3498090850438947e-06
loss T_new: 0.1405630111694336
loss truncated: 0.14064133167266846
loss change: 7.832050323486328e-05
[検証成功] rank制限の記録
  元bond=12, 切り捨てなし=64, 切り捨て後=12, 絶対誤差=2.813202e-04, 相対誤差=2.349809e-06, 損失変化=7.832050e-05. 切り捨てはFrobenius最良近似であり、分類損失の最小化ではない。


## 10. rank制限後のcoreをモデルへ書き戻す

前節の `cores_truncated` は、対象 `TTLinear` の外にある core 列である。この節で `assign_tt_linear_cores` により、その列を `model` の対象層へ書き戻す。

同じ shape の core は既存 Parameter へ値をコピーする。bond 次元が変わり shape が違う core は、その位置の Parameter を差し替える。bias、対象 pair 以外の core、他の層には触らない。このNotebookは optimizer を持ち越さない。shape が変わって Parameter が差し替わっても、optimizer state の移行はしない。

書き戻し前の state を保存し、次を確認する。

- 対象 pair の2 core が、再分割した左右 core と一致する。
- 対象 pair 以外の TT core、`fc1.bias`、`fc2`、`fc3` など、対象 pair 以外の parameter が完全一致する。
- 書き戻し後の通常の `model(inputs)` の logits と loss が、一時 core forward の `logits_truncated`、`loss_truncated` と許容差内で一致する。
- 対象 bond rank が設定した上限 $r_{\mathrm{keep}}$ 以下である。
- 対象 pair 以外の parameter は凍結されたままである。
- 最終モデルの全 parameter が有限値である。

対象 pair の core 値は、SVD の gauge と勾配更新で変わり得る。正しさは、再分割した core との一致と、モデル出力の一致で見る。勾配が非ゼロであること、loss が下がること、accuracy が上がることは要求しない。

このNotebookを最後まで実行した時点の `model` は、rank制限後の2 coreを書き戻した局所更新済みモデルである。checkpoint、CSV、test 評価は保存しない。


In [43]:
# --- rank制限後の core を対象 TTLinear へ書き戻す ---
# 一時 forward の cores_truncated を、モデルの Parameter にする。
cores_before_assign = [core.detach().clone() for core in layer.cores]
state_before_assign = copy.deepcopy(model.state_dict())
pair_names = {
    f"{LAYER_NAME}.cores.{PAIR_INDEX}",
    f"{LAYER_NAME}.cores.{PAIR_INDEX + 1}",
}
assign_tt_linear_cores(layer, cores_truncated)

left_match = torch.equal(layer.cores[PAIR_INDEX], split_truncated.left_core)
right_match = torch.equal(layer.cores[PAIR_INDEX + 1], split_truncated.right_core)
final_bond = int(layer.cores[PAIR_INDEX].shape[-1])
bond_ok = final_bond <= r_keep and int(layer.cores[PAIR_INDEX + 1].shape[0]) == final_bond
finite_ok = all(bool(torch.isfinite(parameter).all()) for parameter in model.parameters())
record(
    "最終モデルへのcore書き戻し",
    "検証成功" if left_match and right_match and bond_ok and finite_ok else "検証失敗",
    f"左core一致={left_match}, 右core一致={right_match}, "
    f"最終bond={final_bond}, 上限 r_keep={r_keep}, 全parameter有限={finite_ok}",
)

other_cores_ok = all(
    torch.equal(layer.cores[index], cores_before_assign[index])
    for index in range(len(layer.cores))
    if index not in {PAIR_INDEX, PAIR_INDEX + 1}
)
state_after_assign = model.state_dict()
other_parameters_ok = all(
    torch.equal(state_before_assign[name], state_after_assign[name])
    for name in state_before_assign
    if name not in pair_names
)
still_frozen = all(
    not parameter.requires_grad
    for name, parameter in model.named_parameters()
    if name not in pair_names
)
record(
    "対象pair以外のstate不変",
    "検証成功" if other_cores_ok and other_parameters_ok and still_frozen else "検証失敗",
    f"対象外core一致={other_cores_ok}, 対象外parameter一致={other_parameters_ok}, "
    f"対象外parameterの凍結={still_frozen}",
)

with torch.no_grad():
    logits_final = model(images)
    loss_final = F.cross_entropy(logits_final, labels)
try:
    torch.testing.assert_close(logits_final, logits_truncated, **tolerances)
except AssertionError as error:
    record("最終モデルと一時forwardのlogits一致", "検証失敗", str(error))
logits_gap = float((logits_final - logits_truncated).abs().max())
record(
    "最終モデルと一時forwardのlogits一致",
    "検証成功",
    f"max_abs={logits_gap:.3e}。許容は atol={tolerances['atol']:.1e}, "
    f"rtol={tolerances['rtol']:.1e} * |logit|",
)
loss_gap_final = abs(float(loss_final - loss_truncated))
record(
    "最終モデルと一時forwardのloss一致",
    "検証成功" if loss_gap_final <= tolerances["atol"] else "検証失敗",
    f"最終loss={float(loss_final):.6e}, 一時forward={float(loss_truncated):.6e}, "
    f"|差|={loss_gap_final:.3e}, 許容={tolerances['atol']:.1e}",
)
print("final TT-rank:", final_bond)
print("final minibatch loss:", float(loss_final))
print(
    "この時点の model は、rank制限後の2 coreを対象TTLinearへ書き戻した局所更新済みモデルである。"
)


[検証成功] 最終モデルへのcore書き戻し
  左core一致=True, 右core一致=True, 最終bond=12, 上限 r_keep=12, 全parameter有限=True
[検証成功] 対象pair以外のstate不変
  対象外core一致=True, 対象外parameter一致=True, 対象外parameterの凍結=True
[検証成功] 最終モデルと一時forwardのlogits一致
  max_abs=0.000e+00。許容は atol=2.0e-04, rtol=2.0e-04 * |logit|
[検証成功] 最終モデルと一時forwardのloss一致
  最終loss=1.406413e-01, 一時forward=1.406413e-01, |差|=0.000e+00, 許容=2.0e-04
final TT-rank: 12
final minibatch loss: 0.14064133167266846
この時点の model は、rank制限後の2 coreを対象TTLinearへ書き戻した局所更新済みモデルである。


In [44]:
# --- 検証結果の一覧 ---
print("項目 | 状態 | 内容")
for row in checks:
    print(f"{row['status']} | {row['item']} | {row['detail']}")
failed = [row["item"] for row in checks if row["status"] == "検証失敗"]
skipped = [row["item"] for row in checks if row["status"] in {"未実行", "SKIP"}]
print("検証失敗:", failed if failed else "なし")
print("未実行またはSKIP:", skipped if skipped else "なし")
print("確認した項目数:", len(checks))
print()
by_item = {row["item"]: row["status"] for row in checks}
print("最終モデルへのcore書き戻し:", by_item["最終モデルへのcore書き戻し"])
print("対象pair以外のstate不変:", by_item["対象pair以外のstate不変"])
print("最終モデルと一時forwardのlogits一致:", by_item["最終モデルと一時forwardのlogits一致"])
print("最終モデルと一時forwardのloss一致:", by_item["最終モデルと一時forwardのloss一致"])
print("最終TT-rank:", final_bond)
print("最終minibatch loss:", float(loss_final))
print(
    "Notebookを最後まで実行した時点の model は、"
    "rank制限後の2 coreを対象TTLinearへ書き戻した局所更新済みモデルである。"
)


項目 | 状態 | 内容
検証成功 | 初期状態の準備 | dtype=torch.float32, device=cuda, ranks=(1, 12, 12, 1), loss=0.140657, finite=True
検証成功 | 正準化で重みが不変 | max_abs=6.258e-07, tolerances={'rtol': 0.0002, 'atol': 0.0002}
検証成功 | 正準化でlogitsが不変 | logits max_abs=2.289e-05, loss 0.140657 -> 0.140657
検証成功 | 左直交 | center=0 より左の ||M^T M - I||_F = []（空なら、pairの左にcoreがない）。上限 2.0e-04
検証成功 | 右直交 | centerより右の ||N N^T - I||_F = [6.406932016034261e-07, 7.152992793635349e-07]。上限 2.0e-04
検証成功 | 独立な学習変数T | shape=(1, 32, 2, 8, 8, 12), leaf=True
検証成功 | Tを使うforwardとcore列forwardの一致 | max_abs=9.537e-06, grad_fn=AddmmBackward0, loss=0.140657
検証成功 | Tからlogitsまでの計算グラフ | grad_fn=AddmmBackward0
検証成功 | 勾配gの取得 | shape=(1, 32, 2, 8, 8, 12), finite=True
検証成功 | SGD 1ステップの有限性 | ||g||_F=3.077406e-01, ||T_new-T||_F=3.077670e-04, eta=0.001
検証成功 | 対象外parameterが不変 | parameter数=8, 値の変化あり=0, gradが付いたparameter=[]
検証成功 | 更新後損失の再計算 | L(T)=0.140657, L(T_new)=0.140563. 減少は要求していない。このforwardはno_gradの記録用である。
検証成功 | 切り捨てなしSVDの再結合 | float64 ||M-M_hat||_F=2.719e-